In [2]:
import sqlite3
import spacy
import pandas as pd
from collections import Counter

In [3]:
conn = sqlite3.connect("jobs.db")
cursor = conn.cursor()

In [7]:
query = """
    SELECT * 
    FROM jobspy
    WHERE country='Spain'
    AND id IN (
        SELECT id
        FROM searchterms
        WHERE search_term='data scientist'
        )
"""

df = pd.read_sql(query, conn)

In [8]:
def convert_description(text):
    nlp = spacy.load("en_core_web_lg")
    doc = nlp(text)
    words = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]  # Lemmatization & stopword removal
    return words

In [9]:
df['words_lemma'] = df['description'].apply(lambda x: convert_description(x))

In [10]:
df.words_lemma

0      [workato, workato, transform, technology, comp...
1      [workato, workato, transform, technology, comp...
2      [scientific, datum, platform, datum, product, ...
3      [institute, centre, genomic, regulation, crg, ...
4      [visión, general, bruker, enable, scientist, e...
                             ...                        
377    [excited, grow, global, company, year, history...
378    [dow, believe, put, people, passionate, delive...
379    [data, scientist, expert, scib, country, spain...
380    [executive, director, datum, science, scib, co...
381    [join, dlocal, dlocal, enable, big, company, w...
Name: words_lemma, Length: 382, dtype: object

### Single-word counting

In [11]:
counter = Counter()
for desc in df.words_lemma:
    counter += Counter(desc)

In [12]:
counter.most_common()

[('datum', 2668),
 ('team', 1988),
 ('work', 1983),
 ('experience', 1879),
 ('business', 966),
 ('data', 828),
 ('skill', 769),
 ('development', 761),
 ('ai', 718),
 ('solution', 717),
 ('model', 706),
 ('project', 698),
 ('product', 683),
 ('technology', 636),
 ('opportunity', 627),
 ('process', 618),
 ('science', 616),
 ('company', 603),
 ('learning', 598),
 ('develop', 588),
 ('environment', 580),
 ('include', 567),
 ('support', 564),
 ('drive', 560),
 ('knowledge', 560),
 ('design', 555),
 ('role', 549),
 ('ensure', 544),
 ('technical', 536),
 ('scientist', 535),
 ('good', 525),
 ('customer', 506),
 ('engineer', 483),
 ('build', 468),
 ('world', 465),
 ('need', 464),
 ('new', 460),
 ('year', 459),
 ('provide', 458),
 ('engineering', 458),
 ('lead', 452),
 ('help', 451),
 ('global', 450),
 ('strong', 439),
 ('ability', 436),
 ('value', 432),
 ('service', 429),
 ('application', 427),
 ('machine', 413),
 ('tool', 403),
 ('join', 392),
 ('professional', 386),
 ('analysis', 379),
 ('pla

In [13]:
counter['python']

273

In [14]:
counter['gcp']

63

### Bigram analysis

In [15]:
from itertools import pairwise

bigram_counts = Counter()
for desc in df.words_lemma:
    pairs = list(pairwise(desc))
    bigram_counts += Counter(pairs)

In [16]:
bigram_counts.most_common()

[(('machine', 'learning'), 352),
 (('datum', 'scientist'), 316),
 (('datum', 'science'), 233),
 (('equal', 'opportunity'), 176),
 (('year', 'experience'), 152),
 (('good', 'practice'), 147),
 (('computer', 'science'), 137),
 (('cross', 'functional'), 132),
 (('sexual', 'orientation'), 131),
 (('communication', 'skill'), 121),
 (('experience', 'work'), 113),
 (('work', 'closely'), 108),
 (('opportunity', 'employer'), 106),
 (('work', 'environment'), 104),
 (('dow', 'jones'), 104),
 (('generative', 'ai'), 102),
 (('artificial', 'intelligence'), 99),
 (('skill', 'ability'), 98),
 (('team', 'member'), 98),
 (('functional', 'team'), 95),
 (('datum', 'analysis'), 94),
 (('data', 'drive'), 90),
 (('related', 'field'), 89),
 (('datum', 'pipeline'), 84),
 (('data', 'scientist'), 83),
 (('datum', 'engineer'), 80),
 (('high', 'quality'), 80),
 (('ability', 'work'), 79),
 (('experience', 'datum'), 75),
 (('learning', 'model'), 75),
 (('problem', 'solve'), 74),
 (('decision', 'making'), 73),
 (('bi

In [17]:
# Contain python
filtered_bigrams = Counter({k: v for k, v in bigram_counts.items() if 'python' in k})

In [18]:
filtered_bigrams.total()

546

In [19]:
words_to_filter = {"python", "english", "german", "aws", "cloud", "azure", "gcp", "r", "go"}  # Words we want to filter for

filtered_bigrams = Counter({k: v for k, v in bigram_counts.items() if any(word in words_to_filter for word in k)})

In [20]:
filtered_bigrams.most_common()

[(('cloud', 'platform'), 48),
 (('fluent', 'english'), 40),
 (('cloud', 'base'), 37),
 (('english', 'spanish'), 32),
 (('language', 'python'), 31),
 (('otm', 'r'), 29),
 (('aws', 'gcp'), 27),
 (('aws', 'azure'), 26),
 (('sql', 'python'), 24),
 (('python', 'programming'), 22),
 (('experience', 'cloud'), 22),
 (('platform', 'aws'), 21),
 (('fluency', 'english'), 21),
 (('skill', 'english'), 20),
 (('familiarity', 'cloud'), 20),
 (('python', 'r'), 20),
 (('r', 'principle'), 20),
 (('language', 'english'), 20),
 (('proficiency', 'python'), 19),
 (('english', 'language'), 19),
 (('python', 'experience'), 19),
 (('python', 'sql'), 18),
 (('service', 'aws'), 18),
 (('level', 'english'), 17),
 (('experience', 'python'), 17),
 (('write', 'english'), 16),
 (('azure', 'gcp'), 16),
 (('google', 'cloud'), 16),
 (('english', 'level'), 15),
 (('skill', 'python'), 15),
 (('cloud', 'computing'), 15),
 (('gcp', 'azure'), 14),
 (('proficiency', 'english'), 14),
 (('cloud', 'data'), 14),
 (('aws', 'servic

In [ ]:
import networkx as nx
import plotly.graph_objects as go

G = nx.DiGraph()

for (word1, word2), count in filtered_bigrams.items():
    G.add_edge(word1, word2, weight=count)

pos = nx.spring_layout(G, seed=42)
# Extract edge coordinates
edge_x, edge_y, edge_weights = [], [], []
for edge in G.edges(data=True):
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
    edge_weights.append(edge[2]['weight'])

# Create Plotly edge trace
edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=2, color='cornflowerblue'),
    hoverinfo='text',
    text=[f'Count: {w}' for w in edge_weights],
    mode='lines'
)

# Create node trace
node_x, node_y, node_labels = [], [], []
for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_labels.append(node)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=node_labels,
    textposition="top center",
    marker=dict(size=15, color='orangered', line=dict(width=2, color='black')),
    hoverinfo='text'
)

# Generate the figure
fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    title="Bigram Graph Visualization",
    showlegend=False,
    hovermode='closest',
    margin=dict(b=20, l=20, r=20, t=40),
)

fig.show()

In [ ]:
from pyvis.network import Network

# Create a Pyvis network
net = Network(notebook=True, height="500px", width="100%")

# Add nodes and edges
for (word1, word2), count in filtered_bigrams.items():
    word1_tot = Counter({k: v for k, v in filtered_bigrams.items() if word1 in k}).total()
    net.add_node(word1, label=word1, value=word1_tot, title=f"Count: {word1_tot}")
    word2_tot = Counter({k: v for k, v in filtered_bigrams.items() if word2 in k}).total()
    net.add_node(word2, label=word2, value=word2_tot, title=f"Count: {word2_tot}")
    net.add_edge(word1, word2, title=f"Count: {count}", value=count)

# Save and display graph in browser
net.show("bigram_graph.html")

In [ ]:
df['description'].iloc[0]

In [27]:
import torch
from transformers import pipeline
import json

# Load the pre-trained model for zero-shot classification
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
# Define possible labels for classification
candidate_labels = ["company description", "role or task description", "technical expertise or required knowledge"]

for description_text in df['description'].values[4:10]:
    # Split the text into chunks (here using periods as the separator, you can adjust this)
    chunks = description_text.split('.')
    
    # Initialize a dictionary to store results
    classified_text = {
        "company description": [],
        "role or task description": [],
        "technical expertise or required knowledge": []
    }
    
    # Variable to track the current category
    current_category = None
    
    # Process each chunk and classify it
    for chunk in chunks:
        chunk = chunk.strip()  # Clean up leading/trailing spaces
        if not chunk:
            continue
        
        result = classifier(chunk, candidate_labels)
        best_label = result['labels'][0]
        
        # If the category changes, update the current_category and reset the text
        #if current_category != best_label:
        #    current_category = best_label
            
        # Add the chunk to the corresponding category
        classified_text[best_label].append(chunk)
    
    # Convert dictionary to a JSON-like structure (Python dict)
    json_output = json.dumps(classified_text, ensure_ascii=False, indent=2)
    
    # Print the final JSON output
    print(json_output)


Device set to use mps:0


{
  "company description": [],
  "role or task description": [
    "bruker",
    "location: barcelona or zaragoza responsabilidades:  familiarization with specific instrumentation prior to installation from issued documentation and/or participation in the final testing process in the factory  liaise with customers and colleagues in the site planning process prior to installation  inspect, install, set up, test and achieve specifications of systems and accessories at customers’ sites, and deliver basic operator training  carry out breakdown and planned maintenance at customers’ sites  provide technical, software and application related support and assistance to customers and to colleagues, either directly or by remote diagnosis  carry out procedures necessary to validate systems to certifiable standards  provide technical input to the sales team in non-routine sales cases  frequent travel throughout spain is an essential part of the job",
    "travel to customer sites abroad can occur, 

In [22]:
print(f"""Minimun length of description: {min(df['description'].apply(lambda n: len(n.split())))} \n
          maximum: {max(df['description'].apply(lambda n: len(n.split())))}"
        """)

Minimun length of description: 36 

          maximum: 1530"
        


In [23]:
#summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

#for description_text in df['description'].values[4:10]:
#    print(summarizer(description_text, max_length=300, min_length=30, do_sample=False))

In [24]:
oracle = pipeline(model="deepset/roberta-base-squad2")
question_1 = "Which is the field of work or department of the offered position?"
question_2 = "Which are the main responsibilities for the job position described?"
question_3 = "Which are the required technical skills and experience for the job position?"

for description_text in df['description'].values[4:10]:
    print(oracle(question=question_1, context=description_text, max_answer_len=300))
    print(oracle(question=question_2, context=description_text, max_answer_len=300))
    print(oracle(question=question_3, context=description_text, max_answer_len=300))

Device set to use mps:0


{'score': 0.01792718470096588, 'start': 2388, 'end': 2408, 'answer': 'industry or academia'}
{'score': 0.11668083816766739, 'start': 1250, 'end': 1292, 'answer': 'participation in the final testing process'}
{'score': 0.06660237908363342, 'start': 2151, 'end': 2214, 'answer': 'electrical or electronic engineering, chemistry or biochemistry'}
{'score': 0.057732854038476944, 'start': 2381, 'end': 2401, 'answer': 'industry or academia'}
{'score': 0.033846430480480194, 'start': 1142, 'end': 1285, 'answer': 'familiarization with specific instrumentation prior to installation from issued documentation and/or participation in the final testing process'}
{'score': 0.14190460741519928, 'start': 2144, 'end': 2207, 'answer': 'electrical or electronic engineering, chemistry or biochemistry'}
{'score': 0.09230197221040726, 'start': 968, 'end': 983, 'answer': 'project officer'}
{'score': 0.03871304541826248, 'start': 1032, 'end': 1212, 'answer': 'supporting and executing strategic projects within th

In [5]:
from huggingface_hub import login


# Load Hugging Face token
HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
# Replace "hf_xxx" with your actual token
login(token=HF_TOKEN)

In [ ]:
import streamlit as st
from transformers import pipeline
import os
import torch

#HF_TOKEN = os.environ["HUGGINGFACE_TOKEN"]
HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")

def load_model():
    return pipeline(
        "text-generation",
        model="google/gemma-2-9b-it",#"google/gemma-2b",
        token=HF_TOKEN,
        device=-1
    )

nlp = load_model()

text = "Hello! Are you working now?"
response = nlp(text, max_length=100)
print(response[0]['generated_text'])

model-00002-of-00004.safetensors:  54%|#####4    | 2.68G/4.95G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

In [ ]:

inputs = tokenizer(input_text, return_tensors="pt")